In [17]:
import joblib
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse

In [18]:
# Charger modèle
model = joblib.load("Desktop/projet/phishing_URL_model.pkl")

In [21]:
def extract_features(url):
    parsed = urlparse(url)

    # محاولة جلب الصفحة
    html = ""
    soup = None
    try:
        r = requests.get(url, timeout=5, headers={"User-Agent": "Mozilla/5.0"})
        html = r.text
        soup = BeautifulSoup(html, "html.parser")
    except:
        soup = None

    features = {}

    # -------- Features من URL --------
    features["NumDots"] = url.count(".")
    features["SubdomainLevel"] = max(len(parsed.netloc.split(".")) - 2, 0)
    features["PathLevel"] = parsed.path.count("/")
    features["UrlLength"] = len(url)
    features["NumDash"] = url.count("-")
    features["NumDashInHostname"] = parsed.netloc.count("-")
    features["AtSymbol"] = int("@" in url)
    features["TildeSymbol"] = int("~" in url)
    features["NumUnderscore"] = url.count("_")
    features["NumPercent"] = url.count("%")
    features["NumQueryComponents"] = len(parsed.query.split("&")) if parsed.query else 0
    features["NumAmpersand"] = url.count("&")
    features["NumHash"] = url.count("#")
    features["NumNumericChars"] = sum(c.isdigit() for c in url)
    features["NoHttps"] = int(parsed.scheme != "https")

    # -------- Features من HTML --------
    if soup:
        title = soup.title.string if soup.title else ""
        features["MissingTitle"] = int(title == "")

        iframes = soup.find_all("iframe")
        features["IframeOrFrame"] = int(len(iframes) > 0)

        forms = soup.find_all("form")
        features["NumForms"] = len(forms)

        links = soup.find_all("a", href=True)
        total_links = len(links)

        ext_links = 0
        for link in links:
            href = link["href"]
            if href.startswith("http") and parsed.netloc not in href:
                ext_links += 1

        features["PctExtHyperlinks"] = (
            ext_links / total_links if total_links > 0 else 0
        )

    else:
        # إذا ما قدرناش نجيبو الصفحة
        features["MissingTitle"] = 1
        features["IframeOrFrame"] = 1
        features["NumForms"] = 1
        features["PctExtHyperlinks"] = 1

    return features, soup


In [29]:
#TEST
url = input("dakhl URL:")

features, soup = extract_features(url)

X_new = pd.DataFrame([features])

#
X_new = X_new.reindex(columns=model.feature_names_in_, fill_value=0)

prediction = model.predict(X_new)[0]
proba = model.predict_proba(X_new)[0]
max_proba = max(proba)
# if we couldn't fetch the HTML content
if soup is None:
    print("\n I could not inspect the page content")
    #
    if max_proba < 0.7:
        print("The prediction is based only on the URL structure")
        # ila kanet probability is between 0.5 and 0.7 
        if prediction == 0:
            prediction = 1
            print("Warning level automatically raised because the content could not be inspected")

if prediction == 1:
    print(f"\n: This URL might be PHISHING (Confidence: {max_proba:.2%})")
else:
    print(f"\n This URL is likely SAFE (Confidence: {max_proba:.2%}) ")

print("Confidence:", max(proba))

dakhl URL: https://jebtv.work/ma



 I could not inspect the page content
The prediction is based only on the URL structure
Warning level automatically raised because the content could not be inspected

: This URL might be PHISHING (Confidence: 66.50%)
Confidence: 0.665
